# Solar Filament Segmentation 2026 — assembled from `src/`

Generated by `scripts/build_notebook.py` — **do not edit here**; the repo is the source
of truth. One cell per module: config → data → rle → metric → postprocess → dataset →
model → evaluate → pipeline, then the runner.

**Two modes, decided by what is attached.** With a dataset of `fold*_best.pt`
checkpoints attached: inference only. Without one: train grouped folds until the time
budget, save checkpoints + OOF prob maps, sweep post-processing on OOF, write the
fold-ensemble `submission.csv`. A placeholder submission is written first.

Metric: Panoptic Quality (IoU > 0.5 match), reproduced in `metric` from the organiser's
self-evaluation notebook (azimahmadzadeh/self-evaluation-notebook).


In [ ]:
# --- bootstrap: dependencies ------------------------------------------------------------
import subprocess, sys
try:
    import segmentation_models_pytorch  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"],
                   check=False)
import segmentation_models_pytorch as smp
print("smp", smp.__version__)


In [ ]:
# === src/config.py ===
"""Shared constants, environment discovery and logging.

The same modules run in two places: this repo (Apple-silicon Mac, smoke tests
and post-processing sweeps) and a Kaggle GPU kernel assembled from them by
scripts/build_notebook.py. Everything path-like is resolved here.
"""
from __future__ import annotations

import os
import random
import time
from pathlib import Path

import numpy as np
import torch

T0 = time.time()
SEED = 2026


def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


ON_KAGGLE = Path("/kaggle/working").is_dir()
try:
    REPO = Path(__file__).resolve().parents[1]
except NameError:  # notebook cell: no __file__
    REPO = Path.cwd()
WORK = Path("/kaggle/working") if ON_KAGGLE else REPO
CACHE = Path("/tmp/filament_cache") if ON_KAGGLE else REPO / "cache"
RUNS = WORK if ON_KAGGLE else REPO / "runs"

COMP = "filament-segmentation-2026"
DATA_DIRNAME = "MAGFiLO_1.0_Kaggle_2026"
ANN_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"


def find_data_root() -> Path:
    env = os.environ.get("FILAMENT_DATA")
    cands = [Path(env)] if env else []
    cands += [REPO / "data" / DATA_DIRNAME,
              Path("/kaggle/input/competitions") / COMP / DATA_DIRNAME,
              Path("/kaggle/input") / COMP / DATA_DIRNAME]
    for c in cands:
        if (c / "train" / "train_images").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for pat in (f"*/{DATA_DIRNAME}/train/{ANN_NAME}", f"*/*/{DATA_DIRNAME}/train/{ANN_NAME}"):
            hits = sorted(base.glob(pat))
            if hits:
                return hits[0].parent.parent
    raise FileNotFoundError("competition data not found (set FILAMENT_DATA)")


# --- image / crops --------------------------------------------------------------------
H = W = 2048               # native GONG frame; the metric is computed at this size
IMG_SIZE = 1024            # working resolution: filament thickness ~13 px native -> ~6 px
CROP = 512                 # training crop taken from the IMG_SIZE image
POSITIVE_BIAS = 0.7        # fraction of crops centred on a filament pixel

# --- model / training -----------------------------------------------------------------
ARCH = "unet"              # smp architecture: unet | unetpp | fpn | deeplab | manet
ENCODER = "resnet34"
EPOCHS = 32                # v1 (20 epochs): 3 of 5 folds still rising at the last epoch
BATCH = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
POS_WEIGHT = 2.0           # BCE positive weight; ~0.3 % of pixels are filament
SAMPLES_PER_FILE = 4       # crops drawn per file per epoch
WORKERS = 3 if ON_KAGGLE else 4

# --- folds / budget -------------------------------------------------------------------
N_FOLDS = 5                # grouped by physical jpeg (annotator records never split)
MAX_FOLDS = "auto"         # "auto" trains folds until TIME_BUDGET would be exceeded
START_FOLD = 0
TIME_BUDGET = 8.5 * 3600   # seconds; Kaggle GPU sessions are capped at 12 h

# --- post-processing / selection ------------------------------------------------------
EPOCH_THR = 0.5            # threshold used for per-epoch model selection (no TTA)
MIN_AREA = 150             # component area (native px) below which a prediction is dropped
THRESHOLDS = (0.55, 0.65, 0.75)         # final OOF sweep grid (v1 sweeps: plateau 0.6-0.7) ...
MIN_AREAS = (300, 450)                  # ... x this (v1: 300-400 best, 600 too aggressive) ...
MIN_SCORES = (0.8, 0.9)                 # ... x per-component mean-prob floor (v1 OOF: peak 0.90,
                                        #     +0.015 over none; 0.92 already collapses)
CLOSE_KERNEL = 5           # morphological closing before components (v1 OOF: +0.002)
TTA = True                 # 4-way flip TTA for OOF and test

# --- smoke mode -----------------------------------------------------------------------
DEBUG = os.environ.get("FILAMENT_DEBUG", "0") == "1"
MAX_STEPS = 0              # 0 = full epoch
EVAL_FILES = 0             # 0 = whole validation fold
if DEBUG:
    EPOCHS, MAX_STEPS, EVAL_FILES, MAX_FOLDS = 1, 20, 8, 2
    THRESHOLDS, MIN_AREAS, MIN_SCORES = (0.45, 0.55), (150,), (0.0,)
    WORKERS = 0            # macOS spawn cannot pickle notebook-namespace classes; Kaggle forks


In [ ]:
# === src/data.py ===
"""COCO loading, polygon rasterisation and grouped folds.

Terminology
-----------
file   : a physical GONG H-alpha observation, e.g. ``20160213153054Bh.jpeg``.
record : one annotator's pass over a file. COCO ``images[].id`` is
         ``<annotator_id>-<file_stem>``; a file can have 1-3 records.
"""
from __future__ import annotations

import json
import random
from collections import defaultdict
from functools import lru_cache
from pathlib import Path

import numpy as np
from pycocotools import mask as mask_util



@lru_cache(maxsize=1)
def data_root() -> Path:
    return find_data_root()


def train_img_dir() -> Path:
    return data_root() / "train" / "train_images"


def test_img_dir() -> Path:
    return data_root() / "test" / "test_images"


def ann_path() -> Path:
    return data_root() / "train" / ANN_NAME


def _as_list(x):
    return json.loads(x) if isinstance(x, str) else x


def load_coco():
    """Return (records, anns_by_record, records_by_file).

    records         : {record_id: image dict}
    anns_by_record  : {record_id: [annotation, ...]}
    records_by_file : {file_name: [record_id, ...]} (sorted)
    """
    d = json.load(open(ann_path()))
    records = {i["id"]: i for i in d["images"]}
    anns_by_record: dict[str, list] = defaultdict(list)
    for a in d["annotations"]:
        anns_by_record[a["image_id"]].append(a)
    records_by_file: dict[str, list[str]] = defaultdict(list)
    for rid, i in records.items():
        records_by_file[i["file_name"]].append(rid)
    for k in records_by_file:
        records_by_file[k].sort()
    return records, dict(anns_by_record), dict(records_by_file)


def annotation_polygons(a) -> list[list[float]]:
    segs = _as_list(a["segmentation"])
    return [s for s in segs if len(s) >= 6]


def annotation_rle(a, h: int = H, w: int = W):
    """Compressed COCO RLE dict for one annotation (full resolution)."""
    polys = annotation_polygons(a)
    if not polys:
        return mask_util.encode(np.asfortranarray(np.zeros((h, w), np.uint8)))
    return mask_util.merge(mask_util.frPyObjects(polys, h, w))


def annotation_mask(a, h: int = H, w: int = W) -> np.ndarray:
    return mask_util.decode(annotation_rle(a, h, w)).astype(np.uint8)


def record_semantic_mask(anns: list, h: int = H, w: int = W) -> np.ndarray:
    m = np.zeros((h, w), np.uint8)
    for a in anns:
        m |= annotation_mask(a, h, w)
    return m


def train_files() -> list[str]:
    return sorted(p.name for p in train_img_dir().glob("*.jpeg"))


def test_files() -> list[str]:
    return sorted(p.name for p in test_img_dir().glob("*.jpeg"))


def make_folds(files: list[str], n_folds: int = 5, seed: int = 2026) -> dict[str, int]:
    """Deterministic grouped folds: the unit is the physical file, so all
    annotator records of one observation land in the same fold."""
    files = sorted(files)
    rng = random.Random(seed)
    rng.shuffle(files)
    return {f: i % n_folds for i, f in enumerate(files)}


In [ ]:
# === src/rle.py ===
"""Submission RLE helpers (pycocotools compressed RLE, 2048x2048)."""
from __future__ import annotations

import numpy as np
from pycocotools import mask as mask_util



def mask_to_rle(mask: np.ndarray) -> str:
    rle = mask_util.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle["counts"].decode("ascii")


def rle_to_mask(counts: str, h: int = H, w: int = W) -> np.ndarray:
    return mask_util.decode({"size": [h, w], "counts": counts.encode("ascii")}).astype(np.uint8)


def masks_to_rles(masks) -> list:
    """Binary masks -> pycocotools RLE dicts (what the metric consumes)."""
    return [mask_util.encode(np.asfortranarray(m.astype(np.uint8))) for m in masks]


In [ ]:
# === src/metric.py ===
"""Panoptic Quality exactly as in the organiser's self-evaluation notebook.

For every annotator record of an image, an (n_gt, n_pred) IoU matrix is
computed. A pair with IoU > 0.5 is a hit; every hit contributes its IoU to
the TP sum. A prediction with no hit is an FP, a GT filament with no hit an
FN. Counts are pooled over *all records of all images*:

    PQ = sum(IoU of hits) / (TP + 0.5 * FP + 0.5 * FN)
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
from pycocotools import mask as mask_util

IOU_THRESHOLD = 0.5


def iou_matrix(gt_rles: list, pred_rles: list) -> np.ndarray:
    """gt_rles / pred_rles: lists of pycocotools RLE dicts. Returns (n_gt, n_pred)."""
    if not gt_rles or not pred_rles:
        return np.zeros((len(gt_rles), len(pred_rles)), dtype=np.float64)
    # iscrowd flags for the *second* argument; all zero -> plain IoU
    m = mask_util.iou(pred_rles, gt_rles, [0] * len(gt_rles))  # (n_pred, n_gt)
    return np.asarray(m, dtype=np.float64).T


@dataclass
class PQAccumulator:
    tp_ious: list[float] = field(default_factory=list)
    fp: int = 0
    fn: int = 0
    n_records: int = 0

    def add(self, gt_rles: list, pred_rles: list) -> dict:
        self.n_records += 1
        n_gt, n_pred = len(gt_rles), len(pred_rles)
        if n_gt == 0:
            self.fp += n_pred
            return {"tp": 0, "fp": n_pred, "fn": 0}
        if n_pred == 0:
            self.fn += n_gt
            return {"tp": 0, "fp": 0, "fn": n_gt}
        m = iou_matrix(gt_rles, pred_rles)
        hit = m > IOU_THRESHOLD
        ious = m[hit].tolist()
        fp = int((hit.sum(0) == 0).sum())
        fn = int((hit.sum(1) == 0).sum())
        self.tp_ious.extend(ious)
        self.fp += fp
        self.fn += fn
        return {"tp": len(ious), "fp": fp, "fn": fn, "sum_iou": float(sum(ious))}

    @property
    def tp(self) -> int:
        return len(self.tp_ious)

    def summary(self) -> dict:
        denom = self.tp + 0.5 * self.fp + 0.5 * self.fn
        sum_iou = float(sum(self.tp_ious))
        pq = sum_iou / denom if denom > 0 else 0.0
        sq = sum_iou / self.tp if self.tp else 0.0
        rq = self.tp / denom if denom > 0 else 0.0
        return {
            "pq": pq, "sq": sq, "rq": rq,
            "tp": self.tp, "fp": self.fp, "fn": self.fn, "records": self.n_records,
        }


def score_predictions(gt_by_record: dict[str, list], preds_by_file: dict[str, list],
                      records_by_file: dict[str, list[str]], files: list[str]) -> dict:
    """gt_by_record: {record_id: [rle dict]}, preds_by_file: {file_stem: [rle dict]}."""
    acc = PQAccumulator()
    for f in files:
        stem = f.rsplit(".", 1)[0]
        preds = preds_by_file.get(stem, [])
        for rid in records_by_file[f]:
            acc.add(gt_by_record[rid], preds)
    return acc.summary()


In [ ]:
# === src/postprocess.py ===
"""Turn a probability map into filament instances at native resolution."""
from __future__ import annotations

import cv2
import numpy as np



def upsample(prob: np.ndarray) -> np.ndarray:
    """Any-size prob map -> float32 (H, W)."""
    if prob.dtype == np.uint8:
        prob = prob.astype(np.float32) / 255.0
    if prob.shape != (H, W):
        prob = cv2.resize(prob.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
    return prob


def probs_to_instances(prob: np.ndarray, threshold: float = 0.5, min_area: int = 150,
                       close_kernel: int = 0, min_score: float = 0.0,
                       fill_holes: bool = True, dilate: int = 0) -> list[np.ndarray]:
    """Threshold -> optional morphological closing -> 8-connected components
    -> drop components smaller than `min_area` or whose mean probability is
    below `min_score` (a confidence filter on top of the size filter)
    -> optionally grow (dilate > 0) or shrink (dilate < 0) each kept component
    by |dilate| px (boundary-bias probe; applied per component so neighbours
    are not merged).

    Returns a list of uint8 (H, W) masks, largest first. Closing bridges tiny
    gaps so one filament is less likely to be emitted as several fragments
    (fragments fail the IoU > 0.5 match and each costs 0.5 FP).
    """
    prob = upsample(prob)
    binary = (prob >= threshold).astype(np.uint8)
    if close_kernel > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    out = []
    order = np.argsort(-stats[1:, cv2.CC_STAT_AREA]) + 1
    for lab in order:
        if stats[lab, cv2.CC_STAT_AREA] < min_area:
            continue
        x, y, w, h = stats[lab, :4]
        sub = (labels[y:y + h, x:x + w] == lab).astype(np.uint8)
        if min_score > 0 and float(prob[y:y + h, x:x + w][sub > 0].mean()) < min_score:
            continue
        if fill_holes:
            contours, _ = cv2.findContours(sub, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(sub, contours, -1, 1, thickness=cv2.FILLED)
        m = np.zeros(prob.shape, np.uint8)
        if dilate:
            k = abs(dilate)
            y0, x0 = max(y - k, 0), max(x - k, 0)
            y1, x1 = min(y + h + k, prob.shape[0]), min(x + w + k, prob.shape[1])
            pad = np.zeros((y1 - y0, x1 - x0), np.uint8)
            pad[y - y0:y - y0 + h, x - x0:x - x0 + w] = sub
            se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * k + 1, 2 * k + 1))
            grown = cv2.dilate(pad, se) if dilate > 0 else cv2.erode(pad, se)
            if grown.any():
                m[y0:y1, x0:x1] = grown
                out.append(m)
                continue
        m[y:y + h, x:x + w] = sub
        out.append(m)
    return out


def instances_never_empty(prob: np.ndarray, threshold: float, min_area: int,
                          close_kernel: int = 0, min_score: float = 0.0) -> list[np.ndarray]:
    """Like probs_to_instances but guarantees at least one instance so every
    test image has a submission row."""
    inst = probs_to_instances(prob, threshold, min_area, close_kernel, min_score)
    if not inst:
        inst = probs_to_instances(prob, threshold * 0.5, max(min_area // 3, 10), close_kernel)[:1]
    if not inst:
        p = upsample(prob)
        m = np.zeros(p.shape, np.uint8)
        y, x = np.unravel_index(int(p.argmax()), p.shape)
        m[y:y + 4, x:x + 4] = 1
        inst = [m]
    return inst


In [ ]:
# === src/dataset.py ===
"""Cache building and torch datasets over the downscaled cache.

cache/img{S}/<stem>.png        grayscale image downscaled to S x S
cache/test{S}/<stem>.png       same for test images
cache/mask{S}/<record_id>.png  union mask of one annotator record at S x S
cache/gt_rles.pkl              {record_id: [compressed RLE dict, ...]} at 2048 (for PQ)
cache/meta.json                size, records_by_file, folds
"""
from __future__ import annotations

import json
import pickle
import random

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset



def build_cache(size: int = IMG_SIZE, n_folds: int = N_FOLDS) -> dict:
    """Idempotent: skips files that already exist. Returns meta."""
    records, anns, by_file = load_coco()
    img_dir, tst_dir, msk_dir = CACHE / f"img{size}", CACHE / f"test{size}", CACHE / f"mask{size}"
    for d in (img_dir, tst_dir, msk_dir):
        d.mkdir(parents=True, exist_ok=True)
    gt_rles = {}
    for n, f in enumerate(train_files()):
        stem = f.rsplit(".", 1)[0]
        out = img_dir / f"{stem}.png"
        if not out.exists():
            im = cv2.imread(str(train_img_dir() / f), cv2.IMREAD_GRAYSCALE)
            cv2.imwrite(str(out), cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA))
        for rid in by_file[f]:
            a = anns.get(rid, [])
            gt_rles[rid] = [annotation_rle(x) for x in a if not int(x.get("iscrowd", 0))]
            mout = msk_dir / f"{rid}.png"
            if not mout.exists():
                m = record_semantic_mask(a).astype(np.float32)
                m = cv2.resize(m, (size, size), interpolation=cv2.INTER_AREA)
                cv2.imwrite(str(mout), ((m >= 0.5) * 255).astype(np.uint8))
        if n % 100 == 0:
            log(f"cache train {n}")
    for f in test_files():
        out = tst_dir / (f.rsplit(".", 1)[0] + ".png")
        if not out.exists():
            im = cv2.imread(str(test_img_dir() / f), cv2.IMREAD_GRAYSCALE)
            cv2.imwrite(str(out), cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA))
    with open(CACHE / "gt_rles.pkl", "wb") as fh:
        pickle.dump(gt_rles, fh)
    meta = {"size": size, "records_by_file": by_file, "folds": make_folds(train_files(), n_folds, SEED)}
    json.dump(meta, open(CACHE / "meta.json", "w"))
    log(f"cache ready: {len(by_file)} files, {len(gt_rles)} records, {len(test_files())} test")
    return meta


def load_meta() -> dict:
    return json.load(open(CACHE / "meta.json"))


def load_gt_rles() -> dict:
    return pickle.load(open(CACHE / "gt_rles.pkl", "rb"))


def _read(path) -> np.ndarray:
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if im is None:
        raise FileNotFoundError(path)
    return im


def normalise(img: np.ndarray) -> np.ndarray:
    return (img.astype(np.float32) / 255.0 - 0.5) / 0.25


class CropDataset(Dataset):
    """Random crops from cached images. Each __getitem__ picks one annotator
    record at random for the file (label augmentation across annotators)."""

    def __init__(self, files: list[str], meta: dict, crop: int, samples_per_file: int,
                 positive_bias: float = POSITIVE_BIAS):
        self.files, self.meta = files, meta
        self.S, self.crop, self.spf, self.pos = meta["size"], crop, samples_per_file, positive_bias
        self.img_dir, self.msk_dir = CACHE / f"img{self.S}", CACHE / f"mask{self.S}"

    def __len__(self):
        return len(self.files) * self.spf

    def __getitem__(self, i):
        f = self.files[i // self.spf]
        stem = f.rsplit(".", 1)[0]
        rid = random.choice(self.meta["records_by_file"][f])
        img = _read(self.img_dir / f"{stem}.png")
        msk = _read(self.msk_dir / f"{rid}.png") > 127

        c = self.crop
        if random.random() < self.pos and msk.any():
            ys, xs = np.nonzero(msk)
            k = random.randrange(len(ys))
            cy, cx = ys[k] + random.randint(-c // 3, c // 3), xs[k] + random.randint(-c // 3, c // 3)
            y0 = int(np.clip(cy - c // 2, 0, self.S - c))
            x0 = int(np.clip(cx - c // 2, 0, self.S - c))
        else:
            y0, x0 = random.randint(0, self.S - c), random.randint(0, self.S - c)
        img, msk = img[y0:y0 + c, x0:x0 + c], msk[y0:y0 + c, x0:x0 + c]

        # geometric augs: filaments have no canonical orientation on the disk
        if random.random() < 0.5:
            img, msk = img[:, ::-1], msk[:, ::-1]
        if random.random() < 0.5:
            img, msk = img[::-1], msk[::-1]
        k = random.randrange(4)
        img, msk = np.rot90(img, k), np.rot90(msk, k)

        # photometric augs: GONG sites differ in exposure/contrast
        x = img.astype(np.float32) / 255.0
        if random.random() < 0.8:
            x = np.clip(x, 0, 1) ** random.uniform(0.7, 1.4)
        if random.random() < 0.8:
            x = x * random.uniform(0.8, 1.2) + random.uniform(-0.08, 0.08)
        if random.random() < 0.3:
            x = x + np.random.normal(0, random.uniform(0.005, 0.03), x.shape).astype(np.float32)
        x = (np.clip(x, 0, 1) - 0.5) / 0.25
        return (torch.from_numpy(np.ascontiguousarray(x))[None],
                torch.from_numpy(np.ascontiguousarray(msk).astype(np.float32))[None])


class FullImageDataset(Dataset):
    """Whole cached images (S x S) for validation / test inference."""

    def __init__(self, files: list[str], size: int, split: str = "train"):
        self.files = files
        self.dir = CACHE / (f"img{size}" if split == "train" else f"test{size}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        f = self.files[i]
        img = _read(self.dir / (f.rsplit(".", 1)[0] + ".png"))
        return torch.from_numpy(normalise(img))[None], f


In [ ]:
# === src/model.py ===
"""Network, loss, device and TTA forward."""
from __future__ import annotations

import os

import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
        return torch.device("mps")
    return torch.device("cpu")


def build_model(arch: str = "unet", encoder: str = "resnet34", pretrained: bool = True):
    cls = {"unet": smp.Unet, "unetpp": smp.UnetPlusPlus, "fpn": smp.FPN,
           "deeplab": smp.DeepLabV3Plus, "manet": smp.MAnet}[arch]
    return cls(encoder_name=encoder, encoder_weights="imagenet" if pretrained else None,
               in_channels=1, classes=1)


def bce_dice_loss(logits: torch.Tensor, target: torch.Tensor, dice_weight: float = 1.0,
                  pos_weight: float = 1.0) -> torch.Tensor:
    pw = torch.tensor(pos_weight, device=logits.device)
    bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=pw)
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3))
    denom = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = 1 - (2 * inter + 1) / (denom + 1)
    return bce + dice_weight * dice.mean()


@torch.no_grad()
def predict_probs(model, x: torch.Tensor, tta: bool = True) -> torch.Tensor:
    """x: (B,1,S,S) -> sigmoid probs (B,1,S,S), optional 4x flip TTA."""
    out = torch.sigmoid(model(x))
    if not tta:
        return out
    out = out + torch.sigmoid(model(x.flip(-1))).flip(-1)
    out = out + torch.sigmoid(model(x.flip(-2))).flip(-2)
    out = out + torch.sigmoid(model(x.flip(-1, -2))).flip(-1, -2)
    return out / 4


def save_checkpoint(path, model, **meta):
    torch.save({"state_dict": model.state_dict(), **meta}, path)


def load_checkpoint(path, device):
    ck = torch.load(path, map_location=device, weights_only=False)
    model = build_model(ck["arch"], ck["encoder"], pretrained=False).to(device)
    model.load_state_dict(ck["state_dict"])
    model.eval()
    return model, ck


In [ ]:
# === src/evaluate.py ===
"""Model -> probability maps -> instances -> Panoptic Quality."""
from __future__ import annotations

import numpy as np
import torch
from torch.utils.data import DataLoader



@torch.no_grad()
def predict_files(model, files, size, device, split="train", tta=True, batch_size=2):
    """Yield (file_name, prob map float32 at model resolution (size x size))."""
    model.eval()
    dl = DataLoader(FullImageDataset(files, size, split), batch_size=batch_size,
                    num_workers=min(WORKERS, 2), shuffle=False)
    for x, names in dl:
        p = predict_probs(model, x.to(device), tta=tta).float().cpu().numpy()[:, 0]
        for prob, name in zip(p, names):
            yield name, prob


def score_probs(probs: dict, meta: dict, gt_rles: dict, threshold: float, min_area: int,
                close_kernel: int = 0, min_score: float = 0.0, dilate: int = 0) -> dict:
    """probs: {file_name: prob map (any size, float or uint8)} -> PQ summary."""
    acc = PQAccumulator()
    for name, prob in probs.items():
        preds = masks_to_rles(probs_to_instances(prob, threshold, min_area, close_kernel, min_score,
                                                 dilate=dilate))
        for rid in meta["records_by_file"][name]:
            acc.add(gt_rles[rid], preds)
    return acc.summary()


def evaluate(model, files, meta, device, gt_rles, thresholds=(0.5,), min_area=150, tta=False,
             keep_probs=False):
    """Run the model on `files` and score PQ at each threshold.
    Returns ({thr: summary}, {file: uint8 prob map at model size} if keep_probs else None)."""
    accs = {t: PQAccumulator() for t in thresholds}
    kept = {} if keep_probs else None
    for name, prob in predict_files(model, files, meta["size"], device, tta=tta):
        if keep_probs:
            kept[name] = np.round(prob * 255).astype(np.uint8)
        for t in thresholds:
            preds = masks_to_rles(probs_to_instances(prob, t, min_area))
            for rid in meta["records_by_file"][name]:
                accs[t].add(gt_rles[rid], preds)
    return {t: a.summary() for t, a in accs.items()}, kept


In [ ]:
# === src/pipeline.py ===
"""Orchestrator. Same entry point locally and in the Kaggle kernel.

main():  placeholder submission -> cache -> grouped folds
         -> [inference mode if fold*_best.pt checkpoints are attached under /kaggle/input]
         -> else train folds until TIME_BUDGET, saving fold{k}_best.pt + OOF prob maps,
            rewriting submission.csv after every fold so the run is always scoreable
         -> OOF sweep of (threshold, min_area) on all trained folds
         -> fold-ensemble test prediction -> submission.csv
run():   main() inside a crash guard.
"""
from __future__ import annotations

import json
import time
import traceback
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader



# --- always-scoreable ---------------------------------------------------------------------
def write_placeholder(out_dir: Path):
    rows = []
    for f in test_files():
        m = np.zeros((H, W), np.uint8)
        m[1020:1028, 1020:1028] = 1
        rows.append({"filament_id": f"{f.rsplit('.', 1)[0]}_1", "segmentation_rle": mask_to_rle(m)})
    pd.DataFrame(rows).to_csv(out_dir / "submission.csv", index=False)
    log(f"placeholder submission written: {len(rows)} rows")


def write_submission(test_probs: dict, threshold: float, min_area: int, out_dir: Path,
                     close_kernel: int = 0, min_score: float = 0.0) -> pd.DataFrame:
    rows = []
    for name, prob in test_probs.items():
        stem = name.rsplit(".", 1)[0]
        inst = instances_never_empty(prob, threshold, min_area, close_kernel, min_score)
        for k, m in enumerate(inst, 1):
            rows.append({"filament_id": f"{stem}_{k}", "segmentation_rle": mask_to_rle(m)})
    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    assert df.filament_id.is_unique and len(df) >= len(test_probs)
    df.to_csv(out_dir / "submission.csv", index=False)
    log(f"submission.csv: {len(df)} rows over {len(test_probs)} images "
        f"(thr={threshold}, min_area={min_area}, min_score={min_score})")
    return df


def save_test_probs(test_probs: dict, out_dir: Path):
    """Ensemble-mean test prob maps as uint8 PNGs: lets post-processing be re-done
    locally without another kernel run."""
    d = out_dir / "test"
    d.mkdir(exist_ok=True)
    for name, p in test_probs.items():
        cv2.imwrite(str(d / (name.rsplit(".", 1)[0] + ".png")), np.round(p * 255).astype(np.uint8))


# --- training -----------------------------------------------------------------------------
def train_fold(fold: int, meta: dict, device, out_dir: Path, gt_rles: dict, epochs: int = EPOCHS,
               max_steps: int = MAX_STEPS, eval_files: int = EVAL_FILES, arch: str = ARCH,
               encoder: str = ENCODER) -> dict:
    folds = meta["folds"]
    tr = sorted(f for f, k in folds.items() if k != fold)
    va = sorted(f for f, k in folds.items() if k == fold)
    if eval_files:
        va = va[:eval_files]
    log(f"fold {fold}: train {len(tr)} files, val {len(va)} files, {arch}/{encoder}")

    model = build_model(arch, encoder).to(device)
    ds = CropDataset(tr, meta, crop=CROP, samples_per_file=SAMPLES_PER_FILE)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=WORKERS, drop_last=True,
                    persistent_workers=WORKERS > 0)
    steps = min(len(dl), max_steps) if max_steps else len(dl)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=steps,
                                                pct_start=0.1)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    ckpt = out_dir / f"fold{fold}_best.pt"
    best, history = -1.0, []
    for epoch in range(epochs):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for step, (x, y) in enumerate(dl):
            if step >= steps:
                break
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast("cuda", enabled=use_amp):
                loss = bce_dice_loss(model(x), y, pos_weight=POS_WEIGHT)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item(); n += 1
        res, _ = evaluate(model, va, meta, device, gt_rles, (EPOCH_THR,), MIN_AREA, tta=False)
        r = res[EPOCH_THR]
        rec = {"epoch": epoch, "loss": tot / max(n, 1), **r, "time": time.time() - t0}
        history.append(rec)
        log(f"fold {fold} epoch {epoch:2d} loss {rec['loss']:.4f} | PQ {r['pq']:.4f} SQ {r['sq']:.3f} "
            f"RQ {r['rq']:.3f} tp {r['tp']} fp {r['fp']} fn {r['fn']} | {rec['time']:.0f}s")
        if r["pq"] > best:
            best = r["pq"]
            save_checkpoint(ckpt, model, arch=arch, encoder=encoder, img_size=meta["size"], fold=fold,
                            epoch=epoch, pq=best)
        json.dump(history, open(out_dir / f"fold{fold}_history.json", "w"), indent=1)

    # OOF prob maps with the best epoch + TTA, saved as uint8 PNGs for offline sweeps
    model, _ = load_checkpoint(ckpt, device)
    res, probs = evaluate(model, va, meta, device, gt_rles, (EPOCH_THR,), MIN_AREA, tta=TTA,
                          keep_probs=True)
    oof_dir = out_dir / "oof"
    oof_dir.mkdir(exist_ok=True)
    for name, p in probs.items():
        cv2.imwrite(str(oof_dir / (name.rsplit(".", 1)[0] + ".png")), p)
    log(f"fold {fold} done: best epoch PQ {best:.4f}, TTA PQ {res[EPOCH_THR]['pq']:.4f}")
    return {"fold": fold, "ckpt": ckpt, "best_pq": best, "tta_pq": res[EPOCH_THR]["pq"],
            "oof": probs, "val_files": va, "model": model}


def sweep_postprocess(oof: dict, meta: dict, gt_rles: dict) -> tuple[float, int, float, list]:
    table = []
    for t in THRESHOLDS:
        for a in MIN_AREAS:
            for ms in MIN_SCORES:
                s = score_probs(oof, meta, gt_rles, t, a, CLOSE_KERNEL, ms)
                table.append({"thr": t, "min_area": a, "min_score": ms, "close": CLOSE_KERNEL, **s})
                log(f"sweep thr={t} min_area={a} min_score={ms} close={CLOSE_KERNEL}: PQ {s['pq']:.4f} "
                    f"SQ {s['sq']:.3f} RQ {s['rq']:.3f} tp {s['tp']} fp {s['fp']} fn {s['fn']}")
    best = max(table, key=lambda r: r["pq"])
    return best["thr"], best["min_area"], best["min_score"], table


def predict_test(models: list, size: int, device) -> dict:
    """Average prob maps of several models over the test set. {file: float32 (size,size)}."""
    files = test_files()
    acc = {}
    for m in models:
        for name, p in predict_files(m, files, size, device, split="test", tta=TTA):
            acc[name] = acc[name] + p if name in acc else p.astype(np.float32)
    return {k: v / len(models) for k, v in acc.items()}


def find_attached_checkpoints() -> list[Path]:
    base = Path("/kaggle/input")
    if not base.is_dir():
        return []
    return sorted(set(base.glob("*/fold*_best.pt")) | set(base.glob("*/*/fold*_best.pt")))


# --- entry ---------------------------------------------------------------------------------
# local runs get a "local-" prefix so they can never be confused with pulled kernel runs
OUT_DIR = RUNS if ON_KAGGLE else RUNS / time.strftime("local-%Y%m%d-%H%M%S")


def main():
    seed_everything(SEED)
    device = get_device()
    out_dir = OUT_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    log(f"device={device} on_kaggle={ON_KAGGLE} out={out_dir} cache={CACHE}")
    write_placeholder(out_dir)

    meta = build_cache(IMG_SIZE, N_FOLDS)
    gt_rles = load_gt_rles()

    attached = find_attached_checkpoints()
    if attached:
        log(f"inference mode: {len(attached)} checkpoints attached")
        models, cks = [], []
        for p in attached:
            m, ck = load_checkpoint(p, device)
            models.append(m); cks.append(ck)
            log(f"  {p.name}: fold {ck.get('fold')} epoch {ck.get('epoch')} PQ {ck.get('pq', 0):.4f}")
        thr = float(np.median([ck.get("thr", EPOCH_THR) for ck in cks]))
        area = int(np.median([ck.get("min_area", MIN_AREA) for ck in cks]))
        ms = float(np.median([ck.get("min_score", 0.0) for ck in cks]))
        probs = predict_test(models, meta["size"], device)
        save_test_probs(probs, out_dir)
        write_submission(probs, thr, area, out_dir, CLOSE_KERNEL, ms)
        return

    # training mode
    max_folds = N_FOLDS if MAX_FOLDS == "auto" else int(MAX_FOLDS)
    results, test_sum, n_models, fold_time = [], None, 0, 0.0
    for fold in range(START_FOLD, N_FOLDS):
        if len(results) >= max_folds:
            break
        elapsed = time.time() - T0
        if results and elapsed + 1.15 * fold_time > TIME_BUDGET:
            log(f"time budget: {elapsed / 60:.0f} min used, fold takes {fold_time / 60:.0f} min -> stop")
            break
        t0 = time.time()
        r = train_fold(fold, meta, device, out_dir, gt_rles)
        fold_time = time.time() - t0
        results.append(r)
        # keep the run scoreable: ensemble of folds so far
        p = predict_test([r["model"]], meta["size"], device)
        test_sum = p if test_sum is None else {k: test_sum[k] + p[k] for k in p}
        n_models += 1
        write_submission({k: v / n_models for k, v in test_sum.items()}, EPOCH_THR, MIN_AREA, out_dir,
                         CLOSE_KERNEL)
        del r["model"]
        torch.cuda.empty_cache() if device.type == "cuda" else None

    oof = {}
    for r in results:
        oof.update(r["oof"])
    log(f"OOF sweep over {len(oof)} files from {len(results)} folds")
    thr, area, ms, table = sweep_postprocess(oof, meta, gt_rles)
    summary = {"folds": [{k: v for k, v in r.items() if k in ("fold", "best_pq", "tta_pq")} for r in results],
               "sweep": table, "best_thr": thr, "best_min_area": area, "best_min_score": ms}
    json.dump(summary, open(out_dir / "summary.json", "w"), indent=1)
    for r in results:  # stamp the selected post-processing into the checkpoints
        ck = torch.load(r["ckpt"], map_location="cpu", weights_only=False)
        ck.update(thr=thr, min_area=area, min_score=ms)
        torch.save(ck, r["ckpt"])
    test_mean = {k: v / n_models for k, v in test_sum.items()}
    save_test_probs(test_mean, out_dir)
    write_submission(test_mean, thr, area, out_dir, CLOSE_KERNEL, ms)
    log(f"OOF PQ at selected (thr={thr}, min_area={area}, min_score={ms}): "
        f"{max(table, key=lambda x: x['pq'])['pq']:.4f}")


def run():
    """Entry point with the crash guard: a placeholder submission exists from the first
    seconds of main(), so a crash still leaves a scoreable file."""
    try:
        main()
    except Exception:
        traceback.print_exc()
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        if not (OUT_DIR / "submission.csv").exists():
            write_placeholder(OUT_DIR)
        log("run FAILED; see traceback above")
    log("done")


In [ ]:
# --- run ------------------------------------------------------------------------------
run()
